# 05. Prophet Model

В этом ноутбуке строим Prophet-модель с доступными регрессорами. Если Prophet не установлен, ноутбук выводит понятное сообщение и не ломает остальные ячейки.

## 1. Load processed data

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    Prophet = None
    PROPHET_AVAILABLE = False
    print("Prophet is not installed. Install it with pip install prophet and rerun this notebook.")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import evaluate_forecast
from src.visualization import plot_train_test_forecast, save_plot

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
IMAGES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / "daily_sales_series.csv", parse_dates=["date"])
display(df.head())

## 2. Prepare Prophet dataset

In [ ]:
regressor_candidates = ["onpromotion", "is_holiday", "transactions", "dcoilwtico"]
available_regressors = [col for col in regressor_candidates if col in df.columns]

prophet_df = df[["date", "sales_clean"] + available_regressors].rename(
    columns={"date": "ds", "sales_clean": "y"}
).copy()

for regressor in available_regressors:
    if regressor == "dcoilwtico":
        prophet_df[regressor] = prophet_df[regressor].ffill().bfill()
    else:
        prophet_df[regressor] = prophet_df[regressor].fillna(0)

print("Available regressors:", available_regressors)
display(prophet_df.head())

## 3. Add regressors

In [ ]:
if PROPHET_AVAILABLE:
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
    )
    for regressor in available_regressors:
        model.add_regressor(regressor)
    print("Prophet model is configured.")
else:
    print("Prophet is not installed. Install it with pip install prophet and rerun this notebook.")

## 4. Train/test split

In [ ]:
train_prophet = prophet_df.iloc[:-30].copy()
test_prophet = prophet_df.iloc[-30:].copy()

print("Train:", train_prophet["ds"].min(), "-", train_prophet["ds"].max(), train_prophet.shape)
print("Test:", test_prophet["ds"].min(), "-", test_prophet["ds"].max(), test_prophet.shape)

## 5. Prophet forecast

In [ ]:
if PROPHET_AVAILABLE:
    model.fit(train_prophet)
    future_test = test_prophet[["ds"] + available_regressors].copy()
    forecast = model.predict(future_test)
    prophet_forecast = pd.DataFrame(
        {
            "date": test_prophet["ds"].values,
            "actual": test_prophet["y"].values,
            "forecast": np.maximum(forecast["yhat"].to_numpy(), 0),
        }
    )
    display(prophet_forecast.head())
else:
    prophet_forecast = pd.DataFrame()
    print("Prophet is not installed. Install it with pip install prophet and rerun this notebook.")

## 6. Forecast evaluation

In [ ]:
if PROPHET_AVAILABLE:
    prophet_metrics = evaluate_forecast(prophet_forecast["actual"], prophet_forecast["forecast"], "Prophet")
    metrics_path = DATA_PROCESSED / "prophet_metrics.csv"
    forecast_path = DATA_PROCESSED / "prophet_forecast.csv"
    prophet_metrics.to_csv(metrics_path, index=False)
    prophet_forecast.to_csv(forecast_path, index=False)
    display(prophet_metrics)
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved forecast to: {forecast_path}")
else:
    print("Prophet metrics were not calculated because Prophet is not installed.")

## 7. Forecast visualization

In [ ]:
if PROPHET_AVAILABLE:
    train_plot = train_prophet.rename(columns={"ds": "date", "y": "sales"})
    test_plot = test_prophet.rename(columns={"ds": "date", "y": "sales"})
    plot_train_test_forecast(
        train_plot.tail(180),
        test_plot,
        prophet_forecast,
        target_col="sales",
        forecast_col="forecast",
        title="Prophet Forecast vs Actual",
    )
    save_plot(IMAGES / "prophet_forecast.png")
    plt.show()
else:
    print("Prophet forecast plot was not created because Prophet is not installed.")

## 8. Prophet conclusions

После запуска ноутбука добавьте фактические метрики Prophet в `reports/03_model_comparison.md`. Если Prophet не был установлен, финальное сравнение моделей продолжит работу с baseline и SARIMA.